In [1]:
# Install requirements.
%pip install torch transformers datasets tqdm

  Using cached torch-2.11.0-cp314-cp314-win_amd64.whl.metadata (29 kB)
  Using cached transformers-5.4.0-py3-none-any.whl.metadata (32 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.8.0-py3-none-any.whl.metadata (13 kB)
  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.me


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Load and inspect model.
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
)

print(type(model))
print(type(model.model))
print(model.model.embed_tokens)
print(model.model.layers[0])
print(model.model.layers[0].self_attn)
print(model.model.layers[0].mlp)
print(model.model.norm)

print(model.config)
total = sum(p.numel() for p in model.parameters())
print(f"total params: {total/1e9:.2f}B")

print("num layers:", len(model.model.layers))
for i, layer in enumerate(model.model.layers[:4]):
    print(f"\nLayer {i}")
    print(layer)


for name, module in model.named_modules():
    if (
        "model.layers.0" in name
        or "model.layers.1" in name
        or name in ["model.embed_tokens", "model.norm", "lm_head"]
    ):
        print(name, "->", type(module).__name__)

c:\Users\David\Code\School\CSCI 544 - Applied Natural Language Processing\Project\NLPSpring2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 92.05it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


<class 'transformers.models.mistral.modeling_mistral.MistralForCausalLM'>
<class 'transformers.models.mistral.modeling_mistral.MistralModel'>
Embedding(32768, 4096)
MistralDecoderLayer(
  (self_attn): MistralAttention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
  )
  (mlp): MistralMLP(
    (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
    (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
    (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
  (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
)
MistralAttention(
  (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
  (k_proj): 

In [2]:
# Create stabilizer module.
class Stabilizer(nn.Module):
    def __init__(self, hidden_size=4096, bottleneck=256, dropout=0.0):
        super().__init__()
        self.norm = nn.RMSNorm(hidden_size)
        self.down = nn.Linear(hidden_size, bottleneck, bias=False)
        self.act = nn.SiLU()
        self.up = nn.Linear(bottleneck, hidden_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Parameter(torch.zeros(1))  # starts near identity

    def forward(self, x):
        h = self.norm(x)
        h = self.down(h)
        h = self.act(h)
        h = self.up(h)
        h = self.dropout(h)
        return x + self.gate * h

# Wrap a Mistral transformer layer with stabilizers after attention and MLP.
class WrappedMistralLayer(nn.Module):
    def __init__(self, base_layer, hidden_size=4096, bottleneck=256,
                 use_post_attn=True, use_post_mlp=False):
        super().__init__()
        self.base_layer = base_layer
        self.use_post_attn = use_post_attn
        self.use_post_mlp = use_post_mlp

        if use_post_attn:
            self.stabilizer_attn = Stabilizer(hidden_size, bottleneck)
        if use_post_mlp:
            self.stabilizer_mlp = Stabilizer(hidden_size, bottleneck)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_values=None,
        use_cache=False,
        position_embeddings=None,
        **kwargs
    ):
        residual = hidden_states
        hidden_states = self.base_layer.input_layernorm(hidden_states)

        hidden_states, _ = self.base_layer.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            position_embeddings=position_embeddings,
            **kwargs
        )
        hidden_states = residual + hidden_states

        if self.use_post_attn:
            hidden_states = self.stabilizer_attn(hidden_states)

        residual = hidden_states
        hidden_states = self.base_layer.post_attention_layernorm(hidden_states)
        hidden_states = self.base_layer.mlp(hidden_states)
        hidden_states = residual + hidden_states

        if self.use_post_mlp:
            hidden_states = self.stabilizer_mlp(hidden_states)

        return hidden_states

# Wrap the first 8 layers of the Mistral model with stabilizers.
for i in range(8):
    model.model.layers[i] = WrappedMistralLayer(
        model.model.layers[i],
        hidden_size=model.config.hidden_size,
        bottleneck=256,
        use_post_attn=True,
        use_post_mlp=False,
    )

device = next(model.parameters()).device
dtype = next(model.parameters()).dtype

# Make the dtype match.
# model = model.to(device=device, dtype=dtype)

# Freeze all parameters except those in the stabilizers.
for p in model.parameters():
    p.requires_grad = False

for name, p in model.named_parameters():
    if "stabilizer" in name:
        p.requires_grad = True

# Print stats.
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable/1e6:.2f}M")
print(f"total params: {total/1e9:.2f}B")

trainable params: 16.81M
total params: 7.26B


In [3]:
import random
from datasets import load_dataset, concatenate_datasets

class SupervisedDatasetBuilder:
    def __init__(self, tokenizer, perturb_engine=None):
        self.tokenizer = tokenizer
        self.perturb_engine = perturb_engine

    def maybe_perturb(self, text, perturbation_name=None, rate=0.0, clean_mix_prob=0.5):
        """
        Mix clean and noisy examples.
        clean_mix_prob=0.5 means 50% clean, 50% perturbed.
        """
        if perturbation_name is None or rate <= 0:
            return text
        if random.random() < clean_mix_prob:
            return text
        return self.perturb_engine.apply(text, perturbation_name, rate=rate)

    def build_dataset(self, dataset_name, perturbation_name=None, rate=0.1, clean_mix_prob=0.5, split=None):
        dataset_name = dataset_name.lower()

        if dataset_name == "gsm8k":
            ds = load_dataset("openai/gsm8k", "main", split=split or "train")
            return ds.map(lambda x: self._format_gsm8k(x, perturbation_name, rate, clean_mix_prob))

        elif dataset_name == "mmlu":
            target_subsets = [
                "college_computer_science",
                "college_mathematics",
                "college_physics",
                "electrical_engineering",
                "abstract_algebra",
                "machine_learning",
                "philosophy",
                "high_school_european_history",
                "professional_law",
                "business_ethics"
            ]
            parts = []
            for sub in target_subsets:
                try:
                    # Many MMLU configs have dev/validation/test. Use dev for training.
                    part = load_dataset("cais/mmlu", sub, split=split or "dev")
                    parts.append(part)
                except Exception as e:
                    print(f"Skipping MMLU subset {sub}: {e}")
            ds = concatenate_datasets(parts)
            return ds.map(lambda x: self._format_mmlu(x, perturbation_name, rate, clean_mix_prob))

        elif dataset_name == "arc":
            ds = load_dataset("allenai/ai2_arc", "ARC-Challenge", split=split or "train")
            return ds.map(lambda x: self._format_arc(x, perturbation_name, rate, clean_mix_prob))

        elif dataset_name == "squad":
            ds = load_dataset("rajpurkar/squad_v2", split=split or "train")
            return ds.map(lambda x: self._format_squad(x, perturbation_name, rate, clean_mix_prob))

        elif dataset_name == "bbh":
            # BBH is mainly eval-oriented. Not ideal as main training source.
            ds = load_dataset("lukaemon/bbh", "logical_deduction_seven_objects", split=split or "test")
            return ds.map(lambda x: self._format_bbh(x, perturbation_name, rate, clean_mix_prob))

        elif dataset_name == "humaneval":
            # HumanEval is tiny and eval-oriented. Usually not good for training.
            ds = load_dataset("openai_humaneval", split=split or "test")
            return ds.map(lambda x: self._format_humaneval(x, perturbation_name, rate, clean_mix_prob))

        else:
            raise ValueError(f"Unsupported dataset: {dataset_name}")

    def _prompt_from_messages(self, messages):
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    def _full_text_from_messages(self, messages):
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )

    def _format_gsm8k(self, sample, perturbation_name, rate, clean_mix_prob):
        question = self.maybe_perturb(sample["question"], perturbation_name, rate, clean_mix_prob)
        answer = sample["answer"]

        user_messages = [
            {"role": "system", "content": "You are a helpful assistant. Solve the math problem step by step. The last line must be '#### ANSWER'."},
            {"role": "user", "content": question}
        ]

        return {
            "prompt_text": self._prompt_from_messages(user_messages),
            "target_text": answer,
            "dataset_name": "gsm8k"
        }

    def _format_mmlu(self, sample, perturbation_name, rate, clean_mix_prob):
        question = self.maybe_perturb(sample["question"], perturbation_name, rate, clean_mix_prob)
        choices = sample["choices"]
        answer_idx = sample["answer"]
        answer_letter = ["A", "B", "C", "D"][answer_idx]

        formatted_input = f"{question}\n"
        for i, choice in enumerate(choices):
            formatted_input += f"{['A','B','C','D'][i]}. {choice}\n"
        formatted_input += "Answer:"

        target = answer_letter

        user_messages = [
            {"role": "system", "content": "You are a helpful assistant. Choose the correct answer (A, B, C, or D) for the multiple choice question."},
            {"role": "user", "content": formatted_input}
        ]

        return {
            "prompt_text": self._prompt_from_messages(user_messages),
            "target_text": target,
            "dataset_name": "mmlu"
        }

    def _format_arc(self, sample, perturbation_name, rate, clean_mix_prob):
        question = self.maybe_perturb(sample["question"], perturbation_name, rate, clean_mix_prob)
        choices = sample["choices"]

        formatted_input = f"{question}\n"
        for label, text in zip(choices["label"], choices["text"]):
            formatted_input += f"{label}. {text}\n"
        formatted_input += "Answer:"

        target = str(sample["answerKey"]).strip()

        user_messages = [
            {"role": "system", "content": "You are a helpful assistant. Choose the correct answer from the options provided."},
            {"role": "user", "content": formatted_input}
        ]

        return {
            "prompt_text": self._prompt_from_messages(user_messages),
            "target_text": target,
            "dataset_name": "arc"
        }

    def _format_squad(self, sample, perturbation_name, rate, clean_mix_prob):
        context = sample["context"]
        question = self.maybe_perturb(sample["question"], perturbation_name, rate, clean_mix_prob)

        prompt_content = f"Context: {context}\n\nQuestion: {question}"

        answers = sample["answers"]["text"]
        if len(answers) == 0:
            target = "unanswerable"
        else:
            target = answers[0]

        user_messages = [
            {"role": "system", "content": "You are a helpful assistant. Answer the question based ONLY on the context provided. If the question cannot be answered from the context, respond with 'unanswerable'."},
            {"role": "user", "content": prompt_content}
        ]

        return {
            "prompt_text": self._prompt_from_messages(user_messages),
            "target_text": target,
            "dataset_name": "squad"
        }

    def _format_bbh(self, sample, perturbation_name, rate, clean_mix_prob):
        input_text = self.maybe_perturb(sample["input"], perturbation_name, rate, clean_mix_prob)
        target = sample["target"]

        user_messages = [
            {"role": "system", "content": "You are a helpful assistant. Think step by step and then provide the final answer."},
            {"role": "user", "content": f"{input_text}\nAnswer:"}
        ]

        return {
            "prompt_text": self._prompt_from_messages(user_messages),
            "target_text": target,
            "dataset_name": "bbh"
        }

    def _format_humaneval(self, sample, perturbation_name, rate, clean_mix_prob):
        prompt = self.maybe_perturb(sample["prompt"], perturbation_name, rate, clean_mix_prob)

        # HumanEval doesn't provide a full reference solution in the same way.
        # This makes it poor for supervised fine-tuning unless you have solutions.
        user_messages = [
            {"role": "system", "content": "You are a helpful coding assistant. Complete the Python function. Output ONLY the code within markdown code blocks."},
            {"role": "user", "content": prompt}
        ]

        return {
            "prompt_text": self._prompt_from_messages(user_messages),
            "target_text": "",   # placeholder
            "dataset_name": "humaneval"
        }


In [4]:
from torch.utils.data import Dataset

class MaskedSFTDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=1024):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        prompt_text = sample["prompt_text"]
        target_text = sample["target_text"]

        # Add target after prompt. For chat models, a leading space/newline before target is often fine.
        full_text = prompt_text + target_text

        prompt_ids = self.tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        full_enc = self.tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
        )

        input_ids = full_enc["input_ids"]
        attention_mask = full_enc["attention_mask"]

        labels = input_ids.copy()

        prompt_len = min(len(prompt_ids), len(input_ids))
        for i in range(prompt_len):
            labels[i] = -100

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

class DataCollatorForMaskedSFT:
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        input_ids = [f["input_ids"] for f in features]
        attention_mask = [f["attention_mask"] for f in features]
        labels = [f["labels"] for f in features]

        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            attention_mask, batch_first=True, padding_value=0
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=-100
        )

        if self.pad_to_multiple_of is not None:
            seq_len = input_ids.size(1)
            remainder = seq_len % self.pad_to_multiple_of
            if remainder != 0:
                pad_len = self.pad_to_multiple_of - remainder

                input_ids = torch.nn.functional.pad(
                    input_ids, (0, pad_len), value=self.tokenizer.pad_token_id
                )
                attention_mask = torch.nn.functional.pad(
                    attention_mask, (0, pad_len), value=0
                )
                labels = torch.nn.functional.pad(
                    labels, (0, pad_len), value=-100
                )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

In [5]:
import math
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def get_trainable_parameters(model):
    return [p for p in model.parameters() if p.requires_grad]

def print_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable/1e6:.2f}M")
    print(f"total params: {total/1e9:.2f}B")
    print(f"trainable %: {100.0 * trainable / total:.4f}%")

def train_stabilizers(
    model,
    tokenizer,
    train_dataset,
    val_dataset=None,
    batch_size=1,
    grad_accum_steps=16,
    num_epochs=1,
    lr=2e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,
    log_every=10,
    eval_every=None,
    num_workers=2,
):
    device = next(model.parameters()).device
    use_bf16 = (device.type == "cuda")

    # Important for training decoder-only models
    model.train()
    model.config.use_cache = False

    # Optional but helpful for memory
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    collator = DataCollatorForMaskedSFT(tokenizer)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collator,
        num_workers=num_workers,
        pin_memory=True,
    )

    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collator,
            num_workers=num_workers,
            pin_memory=True,
        )

    params = get_trainable_parameters(model)
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)

    total_update_steps = math.ceil(len(train_loader) / grad_accum_steps) * num_epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(total_update_steps, 1)
    )

    print_trainable_parameters(model)

    global_step = 0
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(num_epochs):
        running_loss = 0.0
        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for step, batch in enumerate(progress):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
                outputs = model(**batch)
                loss = outputs.loss / grad_accum_steps

            loss.backward()
            running_loss += loss.item() * grad_accum_steps

            if (step + 1) % grad_accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(params, max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if global_step % log_every == 0:
                    avg_loss = running_loss / log_every
                    running_loss = 0.0
                    lr_now = scheduler.get_last_lr()[0]
                    progress.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{lr_now:.2e}")

                if eval_every is not None and val_loader is not None and global_step % eval_every == 0:
                    val_loss = evaluate_loss(model, val_loader)
                    print(f"\n[eval @ step {global_step}] val_loss={val_loss:.4f}")
                    model.train()

        # Flush partial accumulation at epoch end
        leftover = len(train_loader) % grad_accum_steps
        if leftover != 0:
            torch.nn.utils.clip_grad_norm_(params, max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        if val_loader is not None:
            val_loss = evaluate_loss(model, val_loader)
            print(f"\n[end epoch {epoch+1}] val_loss={val_loss:.4f}")
            model.train()

    model.config.use_cache = True
    return model

@torch.no_grad()
def evaluate_loss(model, dataloader, max_batches=None):
    model.eval()
    device = next(model.parameters()).device
    use_bf16 = (device.type == "cuda")

    total_loss = 0.0
    total_batches = 0

    for i, batch in enumerate(dataloader):
        if max_batches is not None and i >= max_batches:
            break

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
            outputs = model(**batch)
            loss = outputs.loss

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(total_batches, 1)

import random
import string
import re

class PerturbationEngine:
    def __init__(self):
        # Semantic: Homophones map
        self.homophones_map = {
            "their": ["there", "they're"], "there": ["their", "they're"], "they're": ["their", "there"],
            "your": ["you're"], "you're": ["your"], "its": ["it's"], "it's": ["its"],
            "to": ["too", "two"], "too": ["to", "two"], "two": ["to", "too"],
            "then": ["than"], "than": ["then"], "weather": ["whether"], "whether": ["weather"],
            "write": ["right"], "right": ["write"], "read": ["red"], "red": ["read"],
            "for": ["four"], "four": ["for"], "sun": ["son"], "son": ["sun"]
        }

        # Surface: OCR visual lookalikes
        self.ocr_map = {
            'l': '1', '1': 'l', 'I': '1', 'O': '0', '0': 'O', 'S': '5', '5': 'S',
            'B': '8', '8': 'B', 'Z': '2', '2': 'Z', 'c': 'e', 'e': 'c',
            'o': 'a', 'a': 'o', 'i': 'j', 'j': 'i', 'm': 'n', 'n': 'm',
            'v': 'u', 'u': 'v', 'F': 'P', 'P': 'F'
        }

        # Surface: QWERTY keyboard adjacency
        self.qwerty_map = {
            'q': 'wa', 'w': 'qase', 'e': 'wsdr', 'r': 'edft', 't': 'rfgy', 'y': 'tghu', 'u': 'yhji', 'i': 'ujko', 'o': 'iklp', 'p': 'ol',
            'a': 'qwsz', 's': 'qweadz', 'd': 'wserfc', 'f': 'ertdgv', 'g': 'rtyfhb', 'h': 'tygjnm', 'j': 'yhuikm', 'k': 'uijolm', 'l': 'iopk',
            'z': 'asx', 'x': 'zsdc', 'c': 'xdfv', 'v': 'cfgb', 'b': 'vghn', 'n': 'bhjm', 'm': 'njk'
        }

        # Semantic: Speech Fillers
        self.speech_fillers = ["um", "uh", "like", "you know", "er", "ah", "i mean"]

    def apply(self, text, method_name, rate=0.1):
        """Unified interface to apply any perturbation."""
        if rate == 0: return text

        method_name = method_name.lower()
        if method_name == "typos":
            return self._add_typos(text, rate)
        elif method_name == "ocr":
            return self._add_ocr(text, rate)
        elif method_name == "qwerty":
            return self._add_qwerty(text, rate)
        elif method_name == "whitespace":
            return self._add_whitespace_case(text, rate)
        elif method_name == "homophones":
            return self._add_homophones(text, rate)
        elif method_name == "speech":
            return self._add_speech_fillers(text, rate)
        else:
            raise ValueError(f"Unknown perturbation method: {method_name}")

    # --- Surface Level ---
    def _add_typos(self, text, error_rate):
        chars = list(text)
        for i in range(len(chars) - 1, -1, -1):
            if chars[i].isdigit() or chars[i] in string.whitespace: continue
            if random.random() < error_rate:
                r = random.random()
                if r < 0.25 and i < len(chars)-1 and not chars[i+1].isdigit():
                    chars[i], chars[i+1] = chars[i+1], chars[i]
                elif r < 0.50: del chars[i]
                elif r < 0.75: chars[i] = random.choice(string.ascii_lowercase)
                else: chars.insert(i, random.choice(string.ascii_lowercase))
        return "".join(chars)

    def _add_ocr(self, text, error_rate):
        chars = list(text)
        for i in range(len(chars)):
            if chars[i] in self.ocr_map and random.random() < error_rate:
                chars[i] = self.ocr_map[chars[i]]
        return "".join(chars)

    def _add_qwerty(self, text, error_rate):
        chars = list(text)
        for i in range(len(chars)):
            char_lower = chars[i].lower()
            if char_lower in self.qwerty_map and random.random() < error_rate:
                neighbor = random.choice(self.qwerty_map[char_lower])
                chars[i] = neighbor.upper() if chars[i].isupper() else neighbor
        return "".join(chars)

    # --- Token Level ---
    def _add_whitespace_case(self, text, error_rate):
        chars = list(text)
        # Whitespace injection/deletion
        for i in range(len(chars) - 1, 0, -1):
            if random.random() < error_rate:
                if chars[i] == ' ':
                    if random.random() < 0.5: del chars[i]
                else:
                    if random.random() < 0.5: chars.insert(i, ' ')

        # Random Case injection
        result = list("".join(chars))
        for i in range(len(result)):
            if random.random() < error_rate:
                if result[i].islower(): result[i] = result[i].upper()
                elif result[i].isupper(): result[i] = result[i].lower()
        return "".join(result)

    # --- Semantic Level ---
    def _add_homophones(self, text, error_rate):
        def replace_match(m):
            word = m.group(0)
            lower = word.lower()
            if lower not in self.homophones_map or random.random() >= error_rate:
                return word
            choice = random.choice(self.homophones_map[lower])
            if word.isupper(): return choice.upper()
            if word[0].isupper(): return choice.capitalize()
            return choice
        return re.sub(r"\b[A-Za-z']+\b", replace_match, text)

    def _add_speech_fillers(self, text, error_rate):
        words = text.split()
        new_words = []
        for word in words:
            new_words.append(word)
            if random.random() < error_rate:
                new_words.append(random.choice(self.speech_fillers))
        return " ".join(new_words)

In [ ]:
# tokenizer.pad_token is needed for batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

builder = SupervisedDatasetBuilder(tokenizer, perturb_engine=PerturbationEngine())

train_hf = builder.build_dataset(
    dataset_name="gsm8k",
    perturbation_name="ocr",   # try "typos", "ocr", "qwerty", etc.
    rate=0.10,
    clean_mix_prob=0.5,
    split="train",
)

# Small held-out set for quick validation
val_hf = builder.build_dataset(
    dataset_name="gsm8k",
    perturbation_name="ocr",
    rate=0.10,
    clean_mix_prob=0.5,
    split="test",
)

# Remove unused columns to keep memory cleaner
keep_cols = ["prompt_text", "target_text", "dataset_name"]
train_hf = train_hf.remove_columns([c for c in train_hf.column_names if c not in keep_cols])
val_hf = val_hf.remove_columns([c for c in val_hf.column_names if c not in keep_cols])

train_ds = MaskedSFTDataset(train_hf, tokenizer, max_length=768)
val_ds = MaskedSFTDataset(val_hf, tokenizer, max_length=768)

model = train_stabilizers(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    val_dataset=val_ds,
    batch_size=1,
    grad_accum_steps=16,
    num_epochs=1,
    lr=2e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,
    log_every=10,
    eval_every=50,
    num_workers=2,
)

def get_stabilizer_state_dict(model):
    return {
        name: tensor.cpu()
        for name, tensor in model.state_dict().items()
        if "stabilizer" in name
    }

torch.save(get_stabilizer_state_dict(model), "mistral_stabilizers_gsm8k_ocr.pt")

Map: 100%|██████████| 1319/1319 [00:00<00:00, 12384.35 examples/s]


trainable params: 16.81M
total params: 7.26B
trainable %: 0.2314%


Epoch 1/1:   0%|          | 0/7473 [00:00<?, ?it/s]

In [16]:
state = torch.load("mistral_stabilizers_gsm8k_ocr.pt", map_location="cpu")
missing, unexpected = model.load_state_dict(state, strict=False)
print("missing:", len(missing), "unexpected:", len(unexpected))

missing: 291 unexpected: 0
